In [10]:

# # ============================================================
# # CELL 1: Install all dependencies (2026 latest versions)
# # ============================================================

# # Core LangChain ecosystem
# %pip install langchain
# %pip install langchain-community
# %pip install langchain-neo4j       # Official Neo4j LangChain integration
# %pip install langchain-openai
# %pip install langchain-experimental  # LLMGraphTransformer
# %pip install langgraph             # Agentic workflow engine

# # Graph Database
# %pip install neo4j>=5.27.0                   # Neo4j Python driver

# # PDF Processing
# %pip install pymupdf>=1.25.0                 # Fast PDF loader (fitz)
# %pip install pypdf>=4.0.0                    # Alternative PDF loader
# %pip install unstructured[all-docs]  # Layout-aware parsing

# # Graph Visualization
# %pip install yfiles_jupyter_graphs           # Interactive graph in Jupyter
# %pip install networkx                 # Graph algorithms

# # Evaluation
# %pip install ragas                   # RAG evaluation
# %pip install deepeval                # LLM evaluation

# # Utilities
# %pip install python-dotenv
# %pip install tiktoken
# %pip install json-repair
# %pip install tqdm

# print("✅ All dependencies installed successfully")

####  Section 1: Environment Setup & Library Installation

In [11]:
import os
from dotenv import load_dotenv


os.environ["NEO4J_URI"]      = os.getenv("NEO4J_URI")
os.environ["NEO4J_USERNAME"] = os.getenv("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = os.getenv("NEO4J_PASSWORD")

PDF_PATH = r"F:\sourab\graph-rag\data\Understanding_Climate_Change.pdf"
CHUNK_SIZE     = 500
CHUNK_OVERLAP  = 100
EMBED_MODEL    = "text-embedding-3-small" 
LLM_EXTRACT    = "gpt-4o"
LLM_GENERATE    = "gpt-4o"
VECTOR_INDEX   = "pdf_vector_index"
NODE_LABEL     = "Chunk"

print("✅ Configuration loaded")
print(f"   PDF:           {PDF_PATH}")
print(f"   Neo4j URI:     {os.environ['NEO4J_URI']}")
print(f"   Embed Model:   {EMBED_MODEL}")
print(f"   LLM Extract:   {LLM_EXTRACT}")
print(f"   LLM Generate:  {LLM_GENERATE}")

✅ Configuration loaded
   PDF:           F:\sourab\graph-rag\data\Understanding_Climate_Change.pdf
   Neo4j URI:     bolt://localhost:7687
   Embed Model:   text-embedding-3-small
   LLM Extract:   gpt-4o
   LLM Generate:  gpt-4o


#### 📁 Section 2: PDF Loading & Document Processing

In [12]:
! uv pip install pymupdf

Audited 1 package in 125ms


In [13]:
from langchain_community.document_loaders.pdf import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = PDF_PATH
print(f"Loading PDF: {PDF_PATH}")
loader = PyMuPDFLoader(PDF_PATH)
raw_documents = loader.load()

print(f"✅ Loaded {len(raw_documents)} pages from PDF")
print(f"   Sample snippet: {raw_documents[0].page_content[:200]}...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)
documents = text_splitter.split_documents(raw_documents)
print(f"✅ Split into {len(documents)} chunks")
print(f"   Average chunk length: {sum(len(d.page_content) for d in documents) // len(documents)} chars")

Loading PDF: F:\sourab\graph-rag\data\Understanding_Climate_Change.pdf
✅ Loaded 33 pages from PDF
   Sample snippet: Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the plane...
✅ Split into 201 chunks
   Average chunk length: 429 chars


#### 🕸️ Section 3: Knowledge Graph Construction

In [14]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    enhanced_schema=True,
    sanitize=True,
    refresh_schema=True
)

print("✅ Connected to Neo4j")
print("Graph Schema:")
print(graph.schema)

✅ Connected to Neo4j
Graph Schema:
Node properties:
- **Person**
  - `age`: INTEGER Min: 28, Max: 35
  - `name`: STRING Example: "Alice"
  - `role`: STRING Available options: ['Data Scientist', 'Engineer', 'Designer']
- **Movie**
  - `genre`: STRING Available options: ['Sci-Fi']
  - `rating`: FLOAT Min: 8.7, Max: 8.7
  - `title`: STRING Example: "The Matrix"
  - `year`: INTEGER Min: 1999, Max: 1999
  - `id`: STRING Example: "1"
  - `released`: DATE Min: 1964-12-16, Max: 1996-09-15
  - `imdbRating`: FLOAT Min: 2.4, Max: 9.3
- **Genre**
  - `name`: STRING Example: "Adventure"
Relationship properties:
- **WATCHED**
  - `rating`: INTEGER Min: 5, Max: 5
  - `watched_on`: STRING Available options: ['2024-01-15']
- **KNOWS**
  - `since`: INTEGER Min: 2020, Max: 2020
The relationships:
(:Person)-[:WATCHED]->(:Movie)
(:Person)-[:KNOWS]->(:Person)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


#### Entity & Relationship Extraction with LLMGraphTransformer

In [18]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*")
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import AzureChatOpenAI

llm_extractor = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    temperature=0
)
graph_transformer = LLMGraphTransformer(
    llm=llm_extractor,
    allowed_nodes=[
        "Concept",
        "Phenomenon",
        "Substance",
        "Process",
        "Technology",
        "Ecosystem",
        "Organization",
        "Location",
    ],
    allowed_relationships=[
        "CAUSES",
        "CONTRIBUTES_TO",
        "LEADS_TO",
        "IMPACTS",
        "EMITS",
        "ABSORBS",
        "MITIGATES",
        "PART_OF",
        "LOCATED_IN",
        "RELATED_TO"
    ],
    relationship_properties=["confidence"],
    strict_mode=False
)

#### For Async Processing Only

In [20]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*")
import asyncio
import nest_asyncio
from tqdm.auto import tqdm

nest_asyncio.apply()

BATCH_SIZE = 5
MAX_CONCURRENT = 3    # ← Tune this: higher = faster but more rate-limit risk
all_graph_docs = []

async def extract_graph_async(documents):
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    batches = [documents[i:i+BATCH_SIZE] for i in range(0,len(documents), BATCH_SIZE)]
    pbar = tqdm(total=len(batches), desc="Extracting graph (async)")

    async def process_batch(batch):
        async with semaphore:
            result = await graph_transformer.aconvert_to_graph_documents(batch)
            pbar.update(1)
            return result
    
    results = await asyncio.gather(
        *[process_batch(batch)for batch in batches]
    )
    pbar.close()

    all_docs, errors = [],0
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            print(f"⚠️   Batch {i} failed: {result}")
            errors +=1
        else:
            all_docs.extend(result)

    if errors:
        print(f"⚠️   {errors}/{len(batches)} batches failed — check rate limits or reduce MAX_CONCURRENT")

    return all_docs
all_graph_docs = asyncio.run(extract_graph_async(documents))


Extracting graph (async):   0%|          | 0/41 [00:00<?, ?it/s]

In [21]:
print(len(all_graph_docs))

201


In [22]:
all_graph_docs[0].nodes[:5]

[Node(id='Climate Change', type='Concept', properties={}),
 Node(id='Global Climate', type='Concept', properties={}),
 Node(id='Burning Of Fossil Fuels', type='Process', properties={}),
 Node(id='Deforestation', type='Process', properties={})]

In [23]:
all_graph_docs[0].relationships[:5]

[Relationship(source=Node(id='Climate Change', type='Concept', properties={}), target=Node(id='Global Climate', type='Concept', properties={}), type='RELATED_TO', properties={}),
 Relationship(source=Node(id='Burning Of Fossil Fuels', type='Process', properties={}), target=Node(id='Climate Change', type='Concept', properties={}), type='CONTRIBUTES_TO', properties={}),
 Relationship(source=Node(id='Deforestation', type='Process', properties={}), target=Node(id='Climate Change', type='Concept', properties={}), type='CONTRIBUTES_TO', properties={})]